# 06B1 PHA in water (CGenFF, GROMACS)

PHA: CGenFF from CHARMM-GUI Ligand Reader & Modeler (see 05B). Water: CHARMM TIP3P. Ions: SOD/CLA. Engine: GROMACS.

The same steps as `06A1_gaff2_gromacs_pha_in_water.ipynb` (the polymer benchmark method: box, water, ions, run files and MD protocol); only the PHA parameters differ (CGenFF instead of GAFF2), and the non-bonded settings follow CHARMM (1.2 nm cut-offs, vdW force-switch from 1.0 nm). For the enzyme + PHA system with CGenFF, use `06B2_charmm_gromacs_enzyme_polymer_in_water.ipynb`.

Every step that runs GROMACS or writes files is behind a `RUN_…`/`WRITE_…` flag that defaults to `False`.

## §1 Inputs (CHARMM-GUI Ligand Reader & Modeler download)

- `LIGAND_READER_DIR`: the unpacked download (contains `gromacs/` with `LIG.itp` and `charmm36.itp`, and `ligandrm.pdb`). It is only read.
- `SDF_PATH`: the SDF uploaded to CHARMM-GUI; it gives bond orders for the stereocentre check (optional).
- The GROMACS folders `charmm_gromacs/dry_polymer/` and `charmm_gromacs/solvated_polymer/` under `examples/output/md_tests/<SYSTEM>/`, laid out like `gromacs/` in 06A1.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
from iphasimulator.naming import oligomer_name

system_name = oligomer_name("3HO", 4)
md_root = output_root / "md_tests" / system_name
LIGAND_READER_DIR = None  # e.g. Path("/path/to/PHO4_builder_charmm-gui-XXXXXXXXXX")
SDF_PATH = None           # e.g. output_root / "polymer_structures/PHO4_R.sdf"
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gromacs_dir = md_root / "charmm_gromacs"
dry_polymer_dir = gromacs_dir / "dry_polymer"
solvated_polymer_dir = gromacs_dir / "solvated_polymer"

{
    "ligand_reader_dir": LIGAND_READER_DIR,
    "dry_polymer_dir": dry_polymer_dir,
    "solvated_polymer_dir": solvated_polymer_dir,
}

## §2 Prepare the CGenFF PHA for GROMACS

`prepare_charmm_dry_polymer_folder` writes `charmm_gromacs/dry_polymer/` from the Ligand Reader download: `topol.top` with `charmm36.itp` (`[ defaults ] 1 2 yes 1.0 1.0`, CGenFF atom types) and `LIG.itp`, the packaged CHARMM TIP3P/SOD/CLA files included, `step5_input.gro` and `index.ndx`, the same files 06A1's ParmEd conversion writes for a GAFF2 PHA. The next cells check the solvent-free topology (charges, CGenFF atom types, R stereocentres) and can minimise it.

In [ ]:
from iphasimulator.charmmgui_import import prepare_charmm_dry_polymer_folder

RUN_CGENFF_PREPARATION = False

if RUN_CGENFF_PREPARATION:
    prepared = prepare_charmm_dry_polymer_folder(LIGAND_READER_DIR, gromacs_dir)
    prepared
else:
    print("Set RUN_CGENFF_PREPARATION = True to write charmm_gromacs/dry_polymer/ from the Ligand Reader download.")

### Validate the CGenFF topology

In [ ]:
from iphasimulator.charmmgui_import import format_checks, validate_dry_ligand_folder

if (dry_polymer_dir / "topol.top").exists():
    print(format_checks(validate_dry_ligand_folder(dry_polymer_dir, sdf_path=SDF_PATH)))
else:
    print(f"Missing dry topology: {dry_polymer_dir / 'topol.top'}")

### Optional solvent-free minimisation

A sanity check of the CGenFF parameters, not a physical result: `gmx grompp` + `gmx mdrun` on `lig.gro` with `dry_minimization.mdp` in `dry_polymer/` (seconds).

In [ ]:
from iphasimulator.charmmgui_import import run_dry_minimization

RUN_DRY_MINIMIZATION = False

if RUN_DRY_MINIMIZATION:
    result = run_dry_minimization(dry_polymer_dir)
    print(f"Potential energy: {result.potential_energy} kJ/mol; maximum force: {result.maximum_force} kJ/mol/nm")
else:
    print("Set RUN_DRY_MINIMIZATION = True to minimise the solvent-free PHA.")

## §3 Add water and ions

`write_charmm_polymer_solvation_files` writes `charmm_gromacs/solvated_polymer/` with the packaged water/ion files from `src/iphasimulator/data/gromacs_solvation/` (`TIP3_SOL.itp`, `SOD.itp`, `CLA.itp`, `tip3_ions_atomtypes.itp`, `ions.mdp`), the CHARMM polymer mdp files from `src/iphasimulator/data/charmm_gromacs_polymer/`, and the scripts `run_solvate_local.sh`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm`, the same scripts as 06A1. `run_solvate_local.sh` then runs `gmx editconf` (cubic box, 1.2 nm padding), `gmx solvate`, `gmx genion` (neutralise, 0.15 M SOD/CLA) and a `grompp` check of the minimisation input.

In [ ]:
from iphasimulator.charmmgui_import import write_charmm_polymer_solvation_files

WRITE_SOLVATION_FILES = False

if WRITE_SOLVATION_FILES:
    solvation_files = write_charmm_polymer_solvation_files(
        gromacs_dir,
        box_padding_nm=1.2,
        ion_concentration_molar=0.15,
        job_name=f"{system_name}_cgenff_polymer_MD",
    )
    solvation_files
else:
    print("Set WRITE_SOLVATION_FILES = True to write the solvation scripts into charmm_gromacs/solvated_polymer/.")

### Run Local Solvation When Ready

This step writes water, ions, final coordinates, index files, and `minim_grompp.log`. It is disabled by default. The default solvent group is `SOL`; if your `make_ndx` output uses a different name, run the script with `--solvent-group <name>`.


In [ ]:
import subprocess

RUN_GROMACS_SOLVATION = False

if RUN_GROMACS_SOLVATION:
    result = subprocess.run(
        ["bash", "run_solvate_local.sh"],
        cwd=solvated_polymer_dir,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("Set RUN_GROMACS_SOLVATION = True to run GROMACS solvation and ion insertion.")

## §4 Check topology and coordinates

The final `step5_input.gro` must have as many atoms as `topol.top` describes, and the topology must contain the solvent and ion molecules. The CHARMM checks then confirm `[ defaults ]`, charges, CGenFF atom types and R stereocentres in the final coordinates, and the last line confirms the system type (PHA + water + ions, no protein).

In [ ]:
from iphasimulator.simulation_gromacs_runner import (
    validate_gromacs_coordinate_topology_counts,
    validate_gromacs_solvated_topology,
)

if (solvated_polymer_dir / "step5_input.gro").exists():
    count_validation = validate_gromacs_coordinate_topology_counts(solvated_polymer_dir)
    topology_validation = validate_gromacs_solvated_topology(solvated_polymer_dir)
    print(f"Coordinate atoms: {count_validation.coordinate_atom_count}")
    print(f"Expected topology atoms: {count_validation.expected_atom_count}")
    print(f"Counts match: {count_validation.valid}")
    print(topology_validation.molecule_counts)
else:
    print(f"Missing final solvated coordinates: {solvated_polymer_dir / 'step5_input.gro'}")

In [ ]:
from iphasimulator.charmmgui_import import (
    ION_MOLECULES,
    WATER_MOLECULES,
    read_topology_molecule_counts,
    validate_charmm_polymer_water_folder,
)

if (solvated_polymer_dir / "step5_input.gro").exists():
    print(format_checks(validate_charmm_polymer_water_folder(solvated_polymer_dir, sdf_path=SDF_PATH)))
    counts = read_topology_molecule_counts(solvated_polymer_dir / "topol.top")
    names = {name.upper(): count for name, count in counts.items()}
    protein = [name for name in counts if name.upper().startswith("PRO")]
    is_pha_in_water = (
        names.get("LIG", 0) >= 1
        and any(names.get(water, 0) for water in WATER_MOLECULES)
        and any(names.get(ion, 0) for ion in ION_MOLECULES)
        and not protein
    )
    print(f"[{'PASS' if is_pha_in_water else 'FAIL'}] PHA in water (LIG + water + ions, no protein): {counts}")
else:
    print(f"Missing final solvated coordinates: {solvated_polymer_dir / 'step5_input.gro'}")

## §5 Optional local minimisation

Runs `bash run_step6_local.sh` in `solvated_polymer/` (`grompp` + `mdrun` for `step6.0_minimization`).

In [ ]:
from iphasimulator.simulation_gromacs_runner import run_gromacs_local_minimization

RUN_LOCAL_MINIMIZATION = False

if RUN_LOCAL_MINIMIZATION:
    result = run_gromacs_local_minimization(solvated_polymer_dir)
    print(result.stdout)
else:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 minimisation locally in solvated_polymer/.")

## §6 Files for 07 (HPC)

### Production Preparation

This notebook prepares local GROMACS minimisation and engine-specific NVT/NPT/production scripts. Submit or benchmark those scripts from notebook 07 only.


### Generated Polymer Run Stages

The same stages and lengths as 06A1 and the polymer benchmark; only the non-bonded block differs (CHARMM: `rlist = rcoulomb = rvdw = 1.2` nm, `vdw-modifier = Force-switch`, `rvdw_switch = 1.0` nm).

| Stage | What it does | Simulation length |
| --- | --- | ---: |
| `step6.0_minimization` | Energy minimisation of the solvated/ionised box | Not time-based |
| `step6.1_nvt` | Constant-volume temperature equilibration at 300 K | 100 ps |
| `step6.2_npt` | Constant-pressure density equilibration at 300 K and 1 bar | 500 ps |
| `step7_production` | Production MD at 300 K and 1 bar | 100 ns |

In [ ]:
{
    "local_minimization_script": solvated_polymer_dir / "run_step6_local.sh",
    "hpc_script_for_07": solvated_polymer_dir / "run_hpc_equilibration_production.slurm",
    "final_start_structure": solvated_polymer_dir / "step5_input.gro",
    "final_topology": solvated_polymer_dir / "topol.top",
}

## Force fields in the final topology

`[ defaults ]` (`1 2 yes 1.0 1.0` is the CHARMM convention), the CGenFF atom types of the PHA (residue `LIG`), and the `[ molecules ]` section of `solvated_polymer/topol.top`.

In [ ]:
def topology_sections(topology_path):
    """[ defaults ], PHA atom types and [ molecules ] from a GROMACS topology and its includes."""
    includes = [topology_path.parent / line.split('"')[1] for line in topology_path.read_text().splitlines()
                if line.strip().startswith("#include")]
    defaults, pha_types, molecules = None, set(), []
    for path in [topology_path] + [p for p in includes if p.exists()]:
        section, molecule = None, None
        for raw in path.read_text().splitlines():
            line = raw.split(";", 1)[0].strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("["):
                section = line.strip("[] ").lower()
                continue
            fields = line.split()
            if section == "defaults" and defaults is None:
                defaults = " ".join(fields)
            elif section == "moleculetype":
                molecule = fields[0]
            elif section == "atoms" and molecule == "LIG" and len(fields) > 1:
                pha_types.add(fields[1])
            elif section == "molecules" and path == topology_path:
                molecules.append((fields[0], int(fields[1])))
    return defaults, sorted(pha_types), molecules

final_topology = solvated_polymer_dir / "topol.top"
if final_topology.exists():
    defaults, pha_types, molecules = topology_sections(final_topology)
    print("[ defaults ]  nbfunc comb-rule gen-pairs fudgeLJ fudgeQQ")
    print("             ", defaults)
    print("PHA (LIG) atom types (CGenFF):", " ".join(pha_types))
    print("[ molecules ]")
    for name, count in molecules:
        print(f"  {name:<6} {count}")
else:
    print(f"Missing final topology: {final_topology}")